# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [1]:
#!pip install -qU ragas==0.2.10

In [2]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [3]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/poojithavanteddu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/poojithavanteddu/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [4]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [5]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [6]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [7]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [53]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [10]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [11]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'd2f07c'. Skipping!
Property 'summary' already exists in node '73a8f5'. Skipping!
Property 'summary' already exists in node '32d083'. Skipping!
Property 'summary' already exists in node 'da4229'. Skipping!
Property 'summary' already exists in node '03ae37'. Skipping!
Property 'summary' already exists in node 'ad1d2d'. Skipping!
Property 'summary' already exists in node 'df8526'. Skipping!
Property 'summary' already exists in node '684563'. Skipping!
Property 'summary' already exists in node '87f14c'. Skipping!
Property 'summary' already exists in node 'b684a4'. Skipping!
Property 'summary' already exists in node '1bd670'. Skipping!
Property 'summary' already exists in node '482832'. Skipping!
Property 'summary' already exists in node '27adc6'. Skipping!
Property 'summary' already exists in node '195a0b'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/41 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '195a0b'. Skipping!
Property 'summary_embedding' already exists in node 'd2f07c'. Skipping!
Property 'summary_embedding' already exists in node '32d083'. Skipping!
Property 'summary_embedding' already exists in node 'da4229'. Skipping!
Property 'summary_embedding' already exists in node 'ad1d2d'. Skipping!
Property 'summary_embedding' already exists in node '73a8f5'. Skipping!
Property 'summary_embedding' already exists in node '1bd670'. Skipping!
Property 'summary_embedding' already exists in node '03ae37'. Skipping!
Property 'summary_embedding' already exists in node 'df8526'. Skipping!
Property 'summary_embedding' already exists in node '27adc6'. Skipping!
Property 'summary_embedding' already exists in node '684563'. Skipping!
Property 'summary_embedding' already exists in node '87f14c'. Skipping!
Property 'summary_embedding' already exists in node '482832'. Skipping!
Property 'summary_embedding' already exists in node 'b684a4'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 39, relationships: 476)

We can save and load our knowledge graphs as follows.

In [12]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 39, relationships: 476)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [13]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [14]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

#### ✅✅  ANSWER 
SingleHopSpecificQuerySynthesizer makes simple, fact-based questions answerable from one part of the text.
MultiHopAbstractQuerySynthesizer creates complex, general questions that need reasoning across multiple parts.
MultiHopSpecificQuerySynthesizer also uses multiple parts of the document but asks more specific questions.
These help evaluate how well a system handles simple facts, detailed facts, and reasoning.
Using all three gives a balanced way to test our RAG pipeline's performance.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [15]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is Volume 2?,"[Chapter 1 Academic Years, Academic Calendars,...",The provided context does not include a defini...,single_hop_specifc_query_synthesizer
1,Could you explain how Chapter 3 relates to the...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of clinical work in a standard term ...,single_hop_specifc_query_synthesizer
2,Could you please explain how the Federal Work-...,[Non-Term Characteristics A program that measu...,The Federal Work-Study (FWS) program is an exc...,single_hop_specifc_query_synthesizer
3,What is included in Appendix A regarding disbu...,[both the credit or clock hours and the weeks ...,Appendix A provides examples illustrating the ...,single_hop_specifc_query_synthesizer
4,What is the role of Title IV in student financ...,[Disbursement Timing in Subscription-Based Pro...,Disbursement timing for Title IV funds in subs...,single_hop_specifc_query_synthesizer
5,How does the impact of term structure on acade...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The impact of term structure on academic progr...,multi_hop_abstract_query_synthesizer
6,How do academic calendars relate to academic y...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",Academic years are defined based on weeks of i...,multi_hop_abstract_query_synthesizer
7,How do scheduled payment periods and installme...,[<1-hop>\n\nboth the credit or clock hours and...,Scheduled payment periods determine the basis ...,multi_hop_abstract_query_synthesizer
8,How do Volume 2 and Volume 8 relate to the req...,[<1-hop>\n\nboth the credit or clock hours and...,Volume 2 discusses the requirements for disbur...,multi_hop_specific_query_synthesizer
9,Wht is the relashunship between volume 8 and t...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [16]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node 'c59d38'. Skipping!
Property 'summary' already exists in node 'f28cce'. Skipping!
Property 'summary' already exists in node '081cfb'. Skipping!
Property 'summary' already exists in node 'cd2eff'. Skipping!
Property 'summary' already exists in node '2e882a'. Skipping!
Property 'summary' already exists in node '7cf400'. Skipping!
Property 'summary' already exists in node '6e0111'. Skipping!
Property 'summary' already exists in node '3c720a'. Skipping!
Property 'summary' already exists in node '846f97'. Skipping!
Property 'summary' already exists in node 'e8bbcd'. Skipping!
Property 'summary' already exists in node '66c3d8'. Skipping!
Property 'summary' already exists in node 'aa03ec'. Skipping!
Property 'summary' already exists in node '91694c'. Skipping!
Property 'summary' already exists in node '78c678'. Skipping!


...

Property 'summary_embedding' already exists in node 'c59d38'. Skipping!
Property 'summary_embedding' already exists in node 'cd2eff'. Skipping!
Property 'summary_embedding' already exists in node '2e882a'. Skipping!
Property 'summary_embedding' already exists in node '6e0111'. Skipping!
Property 'summary_embedding' already exists in node '66c3d8'. Skipping!
Property 'summary_embedding' already exists in node 'f28cce'. Skipping!
Property 'summary_embedding' already exists in node '081cfb'. Skipping!
Property 'summary_embedding' already exists in node 'e8bbcd'. Skipping!
Property 'summary_embedding' already exists in node '78c678'. Skipping!
Property 'summary_embedding' already exists in node '91694c'. Skipping!
Property 'summary_embedding' already exists in node '7cf400'. Skipping!
Property 'summary_embedding' already exists in node '3c720a'. Skipping!
Property 'summary_embedding' already exists in node 'aa03ec'. Skipping!
Property 'summary_embedding' already exists in node '846f97'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [20]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What are the academic years in school programs?,"[Chapter 1 Academic Years, Academic Calendars,...",Chapter 1 Academic Years explains that every e...,single_hop_specifc_query_synthesizer
1,What is 34 CFR 668.3(b) about in terms of week...,[Regulatory Citations Academic year minimums: ...,34 CFR 668.3(b) relates to weeks of instructio...,single_hop_specifc_query_synthesizer
2,What is Volume 8 and how does it relate to the...,[Inclusion of Clinical Work in a Standard Term...,Inclusion of Clinical Work in a Standard Term ...,single_hop_specifc_query_synthesizer
3,What is Title IV and how it related to payment...,[Non-Term Characteristics A program that measu...,Title IV programs are subject to payment perio...,single_hop_specifc_query_synthesizer
4,How do instructional time and program hours af...,[<1-hop>\n\nboth the credit or clock hours and...,"The initial disbursement of student aid funds,...",multi_hop_abstract_query_synthesizer
5,How impact of term structure on academic progr...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in standard ter...,multi_hop_abstract_query_synthesizer
6,Pell TEACH FSEOG loans disbursements rules how...,[<1-hop>\n\nboth the credit or clock hours and...,The context explains that disbursement of Pell...,multi_hop_abstract_query_synthesizer
7,clinical work timing overlapping courses how d...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_abstract_query_synthesizer
8,Volume 8 include in the clinical work in stand...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The context explains that clinical work includ...,multi_hop_specific_query_synthesizer
9,What Volume 2 and Volume 7 info is imp for dis...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...",The context explains that Volume 2 covers acad...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [30]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data1"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data1"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [31]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [32]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [35]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [37]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [38]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [39]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [40]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loans  \n- Direct Unsubsidized Loans  \n- Direct PLUS Loans (including student Federal PLUS Loans and parent Direct PLUS Loans)  \n- Subsidized Federal Stafford Loans  \n- Unsubsidized Federal Stafford Loans  \n- Federal SLS Loans  \n- Federal PLUS Loans (loans made under the Federal Family Education Loan (FFEL) Program before July 1, 2010)\n\nNote that new FFEL Program loans ended effective July 1, 2010, so only existing FFEL loans remain from that program. Additionally, graduate or professional students are only eligible for Direct Unsubsidized Loans, not Direct Subsidized Loans.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [41]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [42]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:
### ✅✅ Answer
qa_evaluator:
This checks if the model’s answer is factually correct by comparing it to the reference answer. It's a general QA accuracy evaluator.

labeled_helpfulness_evaluator:
This checks if the answer is helpful to the user, based on both the question and the correct answer. It evaluates usefulness.

empathy_evaluator:
This checks if the response shows empathy — meaning, does the answer feel kind, understanding, or emotionally appropriate?



## LangSmith Evaluation

In [43]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'giving-character-63' at:
https://smith.langchain.com/o/f74ef342-8d8e-4dcc-aa13-051c9a387f0b/datasets/be6a1c5b-3941-4b8a-8932-3106804051a3/compare?selectedSessions=a9c5405b-44ca-46b2-8bf8-dd629325266a




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Volume 2 and Volume 8 how do they relate to cl...,"Based on the provided context, Volume 8 discus...",None,Volume 2 explains the rules about academic yea...,0,0,0,3.399544,890a2660-43f7-4720-a4f3-eb7e368a5939,4f413a73-ff03-4387-8110-03054d247227
1,How do Volume 7 and Volume 8 relate to disburs...,I don't know.,None,Volume 7 discusses disbursement timing in subs...,0,0,0,0.813967,d60fa909-7cee-4ed9-a9e4-ea557229a501,81a44ff1-383b-4773-be15-32926f6f2480
2,What Volume 2 and Volume 7 info is imp for dis...,I don't know.,None,The context explains that Volume 2 covers acad...,0,0,0,0.660265,d98d7be5-d4d1-4bbf-bd51-3457f0d722e2,2de092e8-31f4-49aa-9f69-9cd90995f7f7
3,Volume 8 include in the clinical work in stand...,"Based on the provided context:\n\nVolume 8, Ch...",None,The context explains that clinical work includ...,1,1,0,4.914592,951e7696-b358-4a86-9867-72614134ea2e,296b9bf2-357b-4754-a52d-194371ef1482
4,clinical work timing overlapping courses how d...,"Based on the provided context, if there are tw...",None,The context explains that clinical work includ...,1,0,0,3.334827,e59fc46b-fea4-439f-a19e-4fdcd1c97460,a5b1e6da-c879-4180-9fb8-dee6b3833dfe
5,Pell TEACH FSEOG loans disbursements rules how...,Based on the provided context:\n\n- TEACH Gran...,None,The context explains that disbursement of Pell...,0,0,0,14.435548,f9a76c51-f613-4878-9b13-0dd3ab0986fb,d9433fa9-3942-44a9-bd50-53e8f06059d3
6,How impact of term structure on academic progr...,"Based on the provided context, the impact of t...",None,The inclusion of clinical work in standard ter...,1,1,0,7.474448,83959e80-37ec-4378-b6f5-aff21e3f3d81,726f0c5e-7457-4a95-a0ed-eb9a72358182
7,How do instructional time and program hours af...,"Based on the provided context, before receivin...",None,"The initial disbursement of student aid funds,...",1,1,0,3.994148,79ff1cbc-a56d-4205-9d69-219ab2611192,f4ef71f1-07ed-49d3-969c-4b5fb6f1626b
8,What is Title IV and how it related to payment...,"Based on the provided context, Title IV refers...",None,Title IV programs are subject to payment perio...,1,1,0,2.651436,a7e2fecf-9a2b-48ba-a779-937c6fe3f070,fc326243-6cea-4564-9437-a9d311d6dfda
9,What is Volume 8 and how does it relate to the...,"Based on the provided context, Volume 8 is a s...",None,Inclusion of Clinical Work in a Standard Term ...,1,0,0,3.268747,55368066-6554-4a6e-93a4-60434857dca2,5e4b963f-76fd-451f-ac1a-001931510dff


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [44]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [45]:
rag_documents = docs

In [46]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

### ✅✅ Answer

Chunk size plays a key role in the performance of a RAG (Retrieval-Augmented Generation) application.

Smaller chunks help retrieve more precise information but may miss the full context needed to answer complex questions.

Larger chunks provide more context but might include irrelevant data, making retrieval less accurate.

Chunk size also affects speed, token usage, and overall answer quality.

Tuning the chunk size helps balance between retrieval precision, context richness, and cost-efficiency.
Optimizing chunk size is essential after improving the core logic ("dope-ifying") of a RAG system.



In [47]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?
### ✅✅ Answer
Changing to a larger embedding model (like text-embedding-3-large) improves how well the system understands and matches text.
It leads to better retrieval of relevant chunks, which boosts answer quality.
Larger models capture deeper meaning but may increase cost and response time.
Overall, it enhances the accuracy and performance of your RAG application.


In [48]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [49]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [50]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [51]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for asking about the kinds of loans available. Based on the information provided, there are several types of federal student loans mentioned:\n\n1. **Direct Subsidized Loans** – These loans are based on the student's financial need and do not accrue interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These are available to students regardless of financial need, and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These are available to parents of dependent students (and to graduate or professional students) to help cover the cost of attendance. There is no fixed loan limit on PLUS Loans, but they cannot exceed the student's cost of attendance minus other financial aid.\n\nAdditionally, if a dependent student's parent cannot obtain a Direct PLUS Loan, the student may be eligible for increased amounts in Direct Unsubsidized Loans.\n\nI hope this helps clarify the loan options available. If you're feeli

Finally, we can evaluate the new chain on the same test set!

In [52]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'loyal-offer-22' at:
https://smith.langchain.com/o/f74ef342-8d8e-4dcc-aa13-051c9a387f0b/datasets/be6a1c5b-3941-4b8a-8932-3106804051a3/compare?selectedSessions=2316ee6c-d2da-4965-8d97-004d3ff5f691




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,Volume 2 and Volume 8 how do they relate to cl...,Thank you for your thoughtful question. From t...,None,Volume 2 explains the rules about academic yea...,0,0,1,4.422066,890a2660-43f7-4720-a4f3-eb7e368a5939,95f5036c-821e-4980-8c63-547fd5677a76
1,How do Volume 7 and Volume 8 relate to disburs...,Thank you for your thoughtful question. I’ve c...,None,Volume 7 discusses disbursement timing in subs...,0,0,1,3.920234,d60fa909-7cee-4ed9-a9e4-ea557229a501,17337bb1-db34-4d77-8933-d95b25b2ccfa
2,What Volume 2 and Volume 7 info is imp for dis...,Thank you for your question—it's clear you're ...,None,The context explains that Volume 2 covers acad...,0,0,1,5.862910,d98d7be5-d4d1-4bbf-bd51-3457f0d722e2,429bb7f7-0856-4d58-9228-e038cf26f88c
3,Volume 8 include in the clinical work in stand...,Thank you for your thoughtful question. I can ...,None,The context explains that clinical work includ...,1,1,1,4.392113,951e7696-b358-4a86-9867-72614134ea2e,9ac0aaa8-9fdd-45b3-a1bc-e6e63d47863d
4,clinical work timing overlapping courses how d...,Thank you for your thoughtful question about h...,None,The context explains that clinical work includ...,1,1,1,3.069298,e59fc46b-fea4-439f-a19e-4fdcd1c97460,2c6cd4f3-6801-4cd0-be3d-f3125f4eb781
5,Pell TEACH FSEOG loans disbursements rules how...,Thank you for your thoughtful question. Based ...,None,The context explains that disbursement of Pell...,0,0,1,3.990237,f9a76c51-f613-4878-9b13-0dd3ab0986fb,99c0dbaa-4afb-4628-8c38-9a48fd0100ea
6,How impact of term structure on academic progr...,Thank you for your thoughtful question. It sou...,None,The inclusion of clinical work in standard ter...,1,1,1,5.522094,83959e80-37ec-4378-b6f5-aff21e3f3d81,3d8c5149-fd43-4915-b592-0fa83ddf9de4
7,How do instructional time and program hours af...,Thank you for your thoughtful question. Based ...,None,"The initial disbursement of student aid funds,...",1,1,1,4.027498,79ff1cbc-a56d-4205-9d69-219ab2611192,1932b69d-5d25-48f5-9db0-c1787d3e986b
8,What is Title IV and how it related to payment...,Thank you for your thoughtful question. Based ...,None,Title IV programs are subject to payment perio...,1,1,1,2.555437,a7e2fecf-9a2b-48ba-a779-937c6fe3f070,8e2b0a9e-50a0-4fd5-88f9-45d68e0776ad
9,What is Volume 8 and how does it relate to the...,Thank you for your thoughtful question. Based ...,None,Inclusion of Clinical Work in a Standard Term ...,0,1,1,1.864944,55368066-6554-4a6e-93a4-60434857dca2,74d5893c-d041-4e15-a5dd-a9c61c32b76e


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.
image.png


Explanation:
I compared two versions of my RAG pipeline. In the second version (loyal-offer-22), I upgraded the embedding model to text-embedding-3-large.
As a result, I noticed improvements in empathy (from 0 to 1.0) and helpfulness (from 0.5 to ~0.7).
This likely happened because the new embedding model produced more meaningful vector representations, helping retrieve more contextually relevant chunks.
That better retrieval led to more emotionally aware and useful answers.
Correctness remained stable, showing that factual accuracy wasn’t compromised.
